# Model comparison for with vs without radiation_day_before

In [2]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

DATA_DIR = Path("../data/NSW")
RESULTS_DIR = Path("../results")

In [3]:
actual = pd.read_csv(DATA_DIR / "nsw_test.csv", parse_dates=["DATETIME"])[["DATETIME", "TOTALDEMAND"]]

raw_temp = pd.read_csv(DATA_DIR / "nsw_features_added.csv", parse_dates=["DATETIME"])[["DATETIME", "TEMPERATURE"]]
actual = actual.merge(raw_temp, on="DATETIME", how="left")

actual.shape

(17520, 3)

In [4]:
# just including the eseasons
season_by_month = {
    12: "Summer", 1: "Summer", 2: "Summer",
    3: "Autumn", 4: "Autumn", 5: "Autumn",
    6: "Winter", 7: "Winter", 8: "Winter",
    9: "Spring", 10: "Spring", 11: "Spring",
}
actual["season"] = actual["DATETIME"].dt.month.map(season_by_month)
actual["temp_band"] = np.where(actual["TEMPERATURE"] >= 18, "above_18", "below_18")

actual.head()

,DATETIME,TOTALDEMAND,TEMPERATURE,season,temp_band
0,2019-01-01 00:00:00,7612.74,22.3,Summer,above_18
1,2019-01-01 00:30:00,7457.58,22.3,Summer,above_18
2,2019-01-01 01:00:00,7243.21,23.0,Summer,above_18
3,2019-01-01 01:30:00,6918.55,23.2,Summer,above_18
4,2019-01-01 02:00:00,6676.58,23.8,Summer,above_18


In [5]:
# just poicking which csvs to look at 

MODEL_STEMS = ["baseline", "lightgbm", "xgboost", "prophet", "random_forest"]

VARIANTS = {
    "with": "{stem}_predictions.csv",
    "without": "{stem}_no_radiation_day_before_predictions.csv",
}

In [6]:
# some helpful functions just so it can calcualte our stats quicky

def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    return {"rmse": rmse, "mae": mae, "mape_pct": mape, "r2": r2}


def segment_stats(model_name, with_radiation, merged):
    segments = {"overall": merged}
    for season in ["Summer", "Autumn", "Winter", "Spring"]:
        segments[season] = merged[merged["season"] == season]
    segments["below_18"] = merged[merged["temp_band"] == "below_18"]
    segments["above_18"] = merged[merged["temp_band"] == "above_18"]

    rows = []
    for segment_name, seg_df in segments.items():
        stats = evaluate(seg_df["TOTALDEMAND"], seg_df["prediction"])
        rows.append({"model_name": model_name, "with_radiation": with_radiation, "segment": segment_name, **stats})
    return rows

In [11]:
# outputs
rows = []

for stem in MODEL_STEMS:
    for variant, pattern in VARIANTS.items():
        path = RESULTS_DIR / pattern.format(stem=stem)
        if not path.exists(): # i have added this here efor now as we dont have day_before for all
            continue

        preds = pd.read_csv(path)
        # just making sure the date joins aren't bugging up
        if preds["DATETIME"].astype(str).str.contains("/").any():
            preds["DATETIME"] = pd.to_datetime(preds["DATETIME"], format="%d/%m/%Y %H:%M")
        else:
            preds["DATETIME"] = pd.to_datetime(preds["DATETIME"], format="%Y-%m-%d %H:%M:%S")
        pred_col = [c for c in preds.columns if c != "DATETIME"][0]
        preds = preds.rename(columns={pred_col: "prediction"})

        merged = actual.merge(preds, on="DATETIME", how="inner")
        assert len(merged) == len(actual), f"{stem} ({variant}) merge dropped rows: {len(merged)} vs {len(actual)}"
        rows.extend(segment_stats(stem, variant == "with", merged))

comparison = pd.DataFrame(rows)
comparison.shape

(56, 7)

In [12]:
# save
comparison.to_csv(RESULTS_DIR / "model_comparison_by_segment.csv", index=False)
comparison

,model_name,with_radiation,segment,rmse,mae,mape_pct,r2
0,baseline,True,overall,535.989120,380.215288,4.743792,0.816091
1,baseline,True,Summer,752.240808,523.904104,6.098386,0.756580
2,baseline,True,Autumn,415.977387,307.099323,4.036791,0.821120
3,baseline,True,Winter,458.510586,357.565800,4.282996,0.844860
4,baseline,True,Spring,452.951988,334.923290,4.584713,0.732202
5,baseline,True,below_18,411.590405,307.569435,3.926999,0.893283
6,baseline,True,above_18,630.381398,448.065665,5.506666,0.741206
7,baseline,False,overall,535.832513,379.974174,4.736390,0.816199
8,baseline,False,Summer,753.088697,527.684088,6.146752,0.756031
9,baseline,False,Autumn,416.707568,307.153469,4.035850,0.820491
